In [21]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn
from autonomy.config import DATA_DIR, MODEL_PATH
from autonomy.model import SteeringModel, get_device
from autonomy.dataset import make_loaders
from autonomy.trainer import Trainer

device = get_device()
print(f"Using {device} device")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using cuda device


In [22]:
# create model (the dummy forward pass inside create() initializes the LazyLinear layer)
model = SteeringModel.create(device)
print(model)

SteeringModel(
  (model): Sequential(
    (0): Conv2d(3, 24, kernel_size=(5, 5), stride=(2, 2))
    (1): ReLU()
    (2): Conv2d(24, 36, kernel_size=(5, 5), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(36, 48, kernel_size=(5, 5), stride=(2, 2))
    (5): ReLU()
    (6): Conv2d(48, 64, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (9): ReLU()
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=107456, out_features=100, bias=True)
    (12): ReLU()
    (13): Linear(in_features=100, out_features=50, bias=True)
    (14): ReLU()
    (15): Linear(in_features=50, out_features=10, bias=True)
    (16): ReLU()
    (17): Linear(in_features=10, out_features=1, bias=True)
    (18): Tanh()
  )
)


In [23]:
# add optimizer (parameters already exist because create() ran a forward pass)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [24]:
# dataset + dataloaders
train_loader, val_loader = make_loaders(DATA_DIR, batch_size=64, val_fraction=0.2, num_workers=0)

1040 train / 259 val samples


In [25]:
# loss + trainer
loss_fn = nn.MSELoss()  # steering is a regression target in [-1, 1], matching the Tanh output
trainer = Trainer(model, optimizer, loss_fn, device)

In [19]:
# check to see if weights first gradients are already 0
# compute gradients for the first batch only (no optimizer step)
model.train()
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)

optimizer.zero_grad()
loss = loss_fn(model(x), y)
loss.backward()

print(f"loss: {loss.item():.6f}")
for name, p in model.named_parameters():
    print(f"{name:20s} grad_norm: {p.grad.norm().item():.6g}")

loss: 0.001521
model.0.weight       grad_norm: 6.12103e-05
model.0.bias         grad_norm: 1.3279e-05
model.2.weight       grad_norm: 0.000229034
model.2.bias         grad_norm: 3.60508e-05
model.4.weight       grad_norm: 0.000307591
model.4.bias         grad_norm: 0.000107408
model.6.weight       grad_norm: 0.000338093
model.6.bias         grad_norm: 0.000257544
model.8.weight       grad_norm: 0.000425358
model.8.bias         grad_norm: 0.000611365
model.11.weight      grad_norm: 0.00945261
model.11.bias        grad_norm: 0.00171465
model.13.weight      grad_norm: 0.000316662
model.13.bias        grad_norm: 0.00512064
model.15.weight      grad_norm: 0.00380466
model.15.bias        grad_norm: 0.0132596
model.17.weight      grad_norm: 0.00641476
model.17.bias        grad_norm: 0.0343145


In [20]:
print(model(x).shape, y.shape)

torch.Size([64, 1]) torch.Size([64, 1])


In [26]:
# training loop, saving the best checkpoint by validation loss
trainer.fit(train_loader, val_loader, epochs=20, ckpt_path=MODEL_PATH)

epoch  1  train 0.02811  val 0.00143
epoch  2  train 0.00236  val 0.00191
epoch  3  train 0.00137  val 0.00127
epoch  4  train 0.00123  val 0.00126
epoch  5  train 0.00120  val 0.00123
epoch  6  train 0.00119  val 0.00121
epoch  7  train 0.00114  val 0.00116
epoch  8  train 0.00109  val 0.00110
epoch  9  train 0.00101  val 0.00104
epoch 10  train 0.00093  val 0.00086
epoch 11  train 0.00075  val 0.00062
epoch 12  train 0.00048  val 0.00034
epoch 13  train 0.00026  val 0.00017
epoch 14  train 0.00018  val 0.00014
epoch 15  train 0.00014  val 0.00012
epoch 16  train 0.00013  val 0.00011
epoch 17  train 0.00011  val 0.00010
epoch 18  train 0.00010  val 0.00008
epoch 19  train 0.00009  val 0.00009
epoch 20  train 0.00008  val 0.00007


7.32237975199394e-05

In [27]:
for name, p in model.named_parameters():
    print(name, p.grad.norm().item())

model.0.weight 0.0023401533253490925
model.0.bias 0.00060567184118554
model.2.weight 0.0030417616944760084
model.2.bias 0.0004915518220514059
model.4.weight 0.0033060002606362104
model.4.bias 0.0008357703918591142
model.6.weight 0.003341218689456582
model.6.bias 0.0017226131167262793
model.8.weight 0.00570816220715642
model.8.bias 0.003271433524787426
model.11.weight 0.0020362837240099907
model.11.bias 9.185265662381425e-05
model.13.weight 0.0056455484591424465
model.13.bias 0.0003341396222822368
model.15.weight 0.0051508937031030655
model.15.bias 0.0008110812050290406
model.17.weight 0.0047104740515351295
model.17.bias 0.0037482953630387783
